# DPO Fine-Tuning Pipeline — 5-Trial Automated Loop
**Track 1, Option A: SFT → DPO (Step 2)**

Platform: Kaggle T4 GPU (16 GB VRAM)
Loads best SFT adapter from Step 1
Dataset: argilla/dpo-mix-7k


In [ ]:
# Cell 1 — Install Dependencies
# !pip install --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
# !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
# !pip install --no-deps unsloth
# !pip install -U bitsandbytes
# !pip install bert_score evaluate nltk pandas tabulate
# print("Dependencies installed.")


In [ ]:
# Cell 2 — Imports
import os, gc, torch
import pandas as pd
import nltk
from datasets import load_dataset
from evaluate import load as load_metric
from bert_score import score as bert_score_fn
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


In [ ]:
# Cell 3 — Configuration
MAX_SEQ_LEN = 512
DTYPE = None
LOAD_IN_4BIT = True
OUTPUT_ROOT = "/kaggle/working"
# Path to the best SFT adapter saved from the SFT notebook
SFT_ADAPTER_PATH = "/kaggle/working/sft_best_adapter"


In [ ]:
# Cell 4 — Same 10 Manual Prompts + Gold References (must match SFT)
EVAL_PROMPTS = [
    "How do I start a small business?",
    "Tell me a joke about programmers.",
    "How can I improve my communication skills?",
    "What is the capital of France?",
    "Explain machine learning like I'm 5.",
    "Give me a fun fact about space.",
    "How do I bake a chocolate cake?",
    "What's the best way to stay motivated?",
    "What are some good daily habits to build?",
    "Tell me about the importance of sleep.",
]

GOLD_REFERENCES = [
    "Start by identifying your niche, writing a business plan, registering your company, and securing initial funding.",
    "Why do programmers prefer dark mode? Because light attracts bugs!",
    "Practice active listening, maintain eye contact, use open body language, and be clear and concise in your speech.",
    "The capital of France is Paris, which is also the country's largest city.",
    "Machine learning is like teaching a computer to learn from examples, just like how you learn to recognize animals from picture books.",
    "A day on Venus is longer than a year on Venus — it takes 243 Earth days to rotate once but only 225 days to orbit the Sun!",
    "Mix flour, sugar, cocoa powder, eggs, butter, and milk together, then bake at 350°F (175°C) for about 30 minutes.",
    "Set small achievable goals, celebrate progress, maintain a routine, and remind yourself of your purpose when motivation dips.",
    "Wake up early, exercise regularly, read for at least 20 minutes, practice gratitude, and plan your day the night before.",
    "Sleep is essential for brain recovery, memory consolidation, immune function, and emotional regulation.",
]


In [ ]:
# Cell 5 — Evaluation Function (BLEU + BERTScore) — Same as SFT
bleu_metric = load_metric("bleu")

def evaluate_model(model, tokenizer, prompts, references, trial_name=""):
    """Generate responses for prompts, compute BLEU and BERTScore F1."""
    FastLanguageModel.for_inference(model)
    model.eval()
    generated = []

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=150,
                                 temperature=0.7, top_p=0.9,
                                 do_sample=True, pad_token_id=tokenizer.pad_token_id)
        response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        generated.append(response)

    bleu_result = bleu_metric.compute(predictions=generated, references=[[r] for r in references])
    bleu_score = bleu_result["bleu"]

    P, R, F1 = bert_score_fn(generated, references, lang="en", verbose=False)
    bert_f1 = F1.mean().item()

    print(f"\n{'='*60}")
    print(f"  Evaluation — {trial_name}")
    print(f"{'='*60}")
    for i, (p, g, r) in enumerate(zip(prompts, generated, references)):
        print(f"\n[Prompt {i+1}]: {p}")
        print(f"  Model:     {g[:200]}")
        print(f"  Reference: {r[:200]}")
    print(f"\n>>> BLEU: {bleu_score:.4f} | BERTScore F1: {bert_f1:.4f}")
    return bleu_score, bert_f1, generated


In [ ]:
# Cell 6 — Load DPO Dataset (argilla/dpo-mix-7k)
raw_dataset = load_dataset("argilla/dpo-mix-7k")

def format_dpo(example):
    """Extract prompt/chosen/rejected from the dataset."""
    prompt = example["prompt"]
    chosen = example["chosen"]
    rejected = example["rejected"]
    # Handle list-of-dicts (conversation) format
    if isinstance(chosen, list):
        chosen = "\n".join([m.get("content", "") for m in chosen if m.get("role") == "assistant"])
    if isinstance(rejected, list):
        rejected = "\n".join([m.get("content", "") for m in rejected if m.get("role") == "assistant"])
    if isinstance(prompt, list):
        prompt = "\n".join([m.get("content", "") for m in prompt if m.get("role") == "user"])
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

dpo_dataset = raw_dataset["train"].map(format_dpo)
dpo_dataset = dpo_dataset.train_test_split(test_size=0.1, seed=42)
dpo_train = dpo_dataset["train"]
dpo_test = dpo_dataset["test"]
print(f"DPO Train: {len(dpo_train)}, DPO Test: {len(dpo_test)}")


In [ ]:
# Cell 7 — Define 5 DPO Trial Configurations
DPO_TRIALS = [
    {"trial_id": 1, "beta": 0.05, "lr": 5e-6, "batch_size": 2, "epochs": 1, "grad_accum": 4},
    {"trial_id": 2, "beta": 0.10, "lr": 1e-5, "batch_size": 4, "epochs": 1, "grad_accum": 2},
    {"trial_id": 3, "beta": 0.20, "lr": 5e-6, "batch_size": 2, "epochs": 2, "grad_accum": 4},
    {"trial_id": 4, "beta": 0.30, "lr": 3e-5, "batch_size": 4, "epochs": 1, "grad_accum": 2},
    {"trial_id": 5, "beta": 0.15, "lr": 7e-6, "batch_size": 2, "epochs": 2, "grad_accum": 4},
]


In [ ]:
# Cell 8 — Automated 5-Trial DPO Loop
results = []

for cfg in DPO_TRIALS:
    trial_id = cfg["trial_id"]
    trial_name = f"DPO Trial {trial_id}"
    output_dir = f"{OUTPUT_ROOT}/dpo_trial_{trial_id}"
    print(f"\n{'#'*60}")
    print(f"  STARTING {trial_name}")
    print(f"  Config: beta={cfg['beta']}, lr={cfg['lr']}, bs={cfg['batch_size']}")
    print(f"{'#'*60}")

    # --- Load best SFT model ---
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=SFT_ADAPTER_PATH,
        max_seq_length=MAX_SEQ_LEN,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Re-enable LoRA training for DPO
    FastLanguageModel.for_training(model)

    # --- DPO Config ---
    dpo_config = DPOConfig(
        beta=cfg["beta"],
        output_dir=output_dir,
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["grad_accum"],
        learning_rate=cfg["lr"],
        num_train_epochs=cfg["epochs"],
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="no",
        warmup_ratio=0.1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        remove_unused_columns=False,
        max_length=MAX_SEQ_LEN,
        max_prompt_length=256,
    )

    # --- DPO Trainer ---
    trainer = DPOTrainer(
        model=model,
        ref_model=None,  # Uses implicit reference (PEFT)
        args=dpo_config,
        train_dataset=dpo_train,
        eval_dataset=dpo_test,
        tokenizer=tokenizer,
    )

    # --- Train ---
    train_result = trainer.train()
    train_loss = train_result.training_loss

    # --- Validation Loss ---
    eval_result = trainer.evaluate()
    val_loss = eval_result.get("eval_loss", float("nan"))

    # --- Evaluate with BLEU + BERTScore on the 10 manual prompts ---
    bleu, bert_f1, gen_responses = evaluate_model(
        model, tokenizer, EVAL_PROMPTS, GOLD_REFERENCES, trial_name
    )

    # --- Save adapter ---
    adapter_path = f"{OUTPUT_ROOT}/dpo_adapter_trial_{trial_id}"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"  Adapter saved to {adapter_path}")

    # --- Record ---
    results.append({
        "Trial": trial_id,
        "Beta": cfg["beta"],
        "LR": cfg["lr"],
        "Batch_Size": cfg["batch_size"],
        "Epochs": cfg["epochs"],
        "Train_Loss": round(train_loss, 4),
        "Val_Loss": round(val_loss, 4),
        "BLEU": round(bleu, 4),
        "BERTScore_F1": round(bert_f1, 4),
    })

    # --- Cleanup ---
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU cache cleared after {trial_name}.")


In [ ]:
# Cell 9 — Results Summary & Best DPO Model
df = pd.DataFrame(results)
print("\n" + "="*60)
print("  DPO TRIAL RESULTS SUMMARY")
print("="*60)
print(df.to_string(index=False))

df["composite"] = df["BLEU"] + df["BERTScore_F1"]
best_idx = df["composite"].idxmax()
best_trial = df.loc[best_idx, "Trial"]
print(f"\n>>> Best DPO Trial: Trial {best_trial}")
print(f"    BLEU={df.loc[best_idx, 'BLEU']}, BERTScore={df.loc[best_idx, 'BERTScore_F1']}")

csv_path = f"{OUTPUT_ROOT}/dpo_trial_results.csv"
df.to_csv(csv_path, index=False)
print(f"Results saved to {csv_path}")


In [ ]:
# Cell 10 — Save Best DPO Model
import shutil

best_src = f"{OUTPUT_ROOT}/dpo_adapter_trial_{best_trial}"
best_dst = f"{OUTPUT_ROOT}/dpo_best_adapter"
if os.path.exists(best_dst):
    shutil.rmtree(best_dst)
shutil.copytree(best_src, best_dst)
print(f"\nBest DPO adapter (SFT+DPO) saved to: {best_dst}")
